In [2]:
import pandas as pd

df = pd.read_json('../data/reddit_raw_dataset.jsonl',lines=True)

In [5]:
df.shape

(8892, 6)

In [9]:
df.columns

Index(['subreddit', 'post_id', 'post_title', 'post_text', 'url', 'comments'], dtype='str')

In [ ]:
f= df['comments'][0][0]
f.keys()

dict_keys(['author', 'body', 'replies'])


In [19]:
records = []

for _, rows in df.iterrows():
    for comment in rows['comments']:
        records.append({
            'subreddit':rows['subreddit'],
            'post_title': rows['post_title'],
            'post_text':rows['post_text'],
            'comments': comment['body']
        })

recordsdf = pd.DataFrame(records)

In [29]:
recordsdf.shape

(181078, 4)

In [ ]:
# any empty or null comments?
print(recordsdf['comments'].isnull().sum())
print((recordsdf['comments'] == '').sum())
# any deleted/removed comments?
print((recordsdf['comments'] == '[deleted]').sum())
print((recordsdf['comments'] == '[removed]').sum())

0
1
0
0


In [63]:
a=['LegalAdviceIndia',
 'AndroidGaming',
 'NetflixBestOf',
 'DisneyPlus',
 'MemeTemplatesOfficial',
 'dankmemes',
 'indieheads',
 'popheads',
 'hiphopheads',
 'Metalcore',
 'Metal',
 'MusicNews']
recordsdf = recordsdf[~recordsdf['subreddit'].isin(a)]
recordsdf.shape

(124950, 4)

In [66]:
recordsdf['c_length'] = recordsdf['comments'].apply(len)
recordsdf['c_length'].describe()

count    124950.000000
mean        120.146443
std         204.278732
min           0.000000
25%          31.000000
50%          62.000000
75%         125.000000
max        7257.000000
Name: c_length, dtype: float64

In [73]:
recordsdf = recordsdf[recordsdf['c_length']>=20]
recordsdf.shape

(108149, 5)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

test = "This movie was absolutely amazing, loved every bit of it!"
analyzer.polarity_scores(test)


{'neg': 0.0, 'neu': 0.483, 'pos': 0.517, 'compound': 0.8612}

In [101]:
def get_sentiment(text):
    return analyzer.polarity_scores(text)['compound']

def label_sentiment(score):
    if score >=0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'
    
recordsdf['title_score']= recordsdf['post_title'].apply(get_sentiment)
recordsdf['text_score']= recordsdf['post_text'].apply(get_sentiment)
recordsdf['comment_score']= recordsdf['comments'].apply(get_sentiment)

recordsdf['title_sentiment']= recordsdf['title_score'].apply(label_sentiment)
recordsdf['text_sentiment']= recordsdf['text_score'].apply(label_sentiment)
recordsdf['comment_sentiment']= recordsdf['comment_score'].apply(label_sentiment)

In [104]:
print(recordsdf['comment_sentiment'].value_counts())
print(recordsdf['title_sentiment'].value_counts())
print(recordsdf['text_sentiment'].value_counts())

comment_sentiment
positive    48656
neutral     33218
negative    26275
Name: count, dtype: int64
title_sentiment
neutral     57348
positive    33179
negative    17622
Name: count, dtype: int64
text_sentiment
positive    51271
neutral     44705
negative    12173
Name: count, dtype: int64


In [107]:
recordsdf.to_csv('../data/indian_reddit_sentiment.csv',index=False)